<a href="https://www.kaggle.com/code/poeticmage/fastsam-with-bounding-box-guided-segmentation?scriptVersionId=314640446" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

## FastSAM with Bounding Box Guided Segmentation
### ultralytics has FastSAMPredictor that guide the FastSAM with the help of bounding box prompt. The coco dataset has images as well as bounding boxes, now we will learn here:
 <h2 style="color:red;">BOUNDING BOX GUIDED SEGMENTATION USING FASTSAM FROM ULTRALYTICS</h2>

## Install depenencies
### These aren't unknown to you.

In [1]:
%%capture
! pip install matplotlib numpy pandas ultralytics pycocotools opencv-python
from ultralytics import FastSAM
from ultralytics.models.fastsam import FastSAMPredictor
from pycocotools.coco import COCO
import torch
import numpy as np
import cv2
import os
import pandas as pd


## Download COCO
### Download the COCO dataset and upload them in your input directory

In [2]:
import os

#create base dir
base = "/kaggle/working/coco"
os.makedirs(base, exist_ok=True)
# move into dir
%cd /kaggle/working/coco
#download
!wget -q http://images.cocodataset.org/zips/val2017.zip
!wget -q http://images.cocodataset.org/annotations/annotations_trainval2017.zip
#unzip
!unzip -q val2017.zip
!unzip -q annotations_trainval2017.zip
#final path
ann_file = "/kaggle/working/coco/annotations/instances_val2017.json"
img_dir  = "/kaggle/working/coco/val2017"
#sanity ck
print("Annotation exists:", os.path.exists(ann_file))
print("Images folder exists:", os.path.exists(img_dir))

/kaggle/working/coco
Annotation exists: True
Images folder exists: True


## Upload data
### Upload the data in your notebooks

In [3]:
coco = COCO(ann_file)
img_ids = list(coco.imgs.keys())

loading annotations into memory...
Done (t=0.50s)
creating index...
index created!


## Override and predict
### Use the over-ride to limit the predictor only to specific task, like conf=0.25 means overlook predictions below 25%, task=segment means do segmentation not detection, etc etc.

In [4]:
overrides = dict(
    conf=0.25,
    task="segment",
    mode="predict",
    model="FastSAM-s.pt",
    imgsz=1024,
    save=False
)

predictor = FastSAMPredictor(overrides=overrides)

os.makedirs("fastsam_output", exist_ok=True)

## Start Bbox guided segmentation
### Start your bounding box guided segmentation over the first 100 images of coco, make sure you resize and interpolate masked image , bbox and original image to same scale otherwise your output is haphazard, aand make sure you remake your bbox from H,W given in image shape

In [5]:
count = 0

TARGET_SIZE = 1024  # must match predictor imgsz

for img_id in img_ids:
    if count >= 100:
        break

    img_info = coco.loadImgs(img_id)[0]
    img_path = os.path.join(img_dir, img_info['file_name'])

    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    H, W = img.shape[:2]

    #  resize image BEFORE starting
    img_resized = cv2.resize(img, (TARGET_SIZE, TARGET_SIZE))
    scale_x = TARGET_SIZE / W
    scale_y = TARGET_SIZE / H

    ann_ids = coco.getAnnIds(imgIds=img_id)
    anns = coco.loadAnns(ann_ids)

    if len(anns) == 0:
        continue

    # run FastSAM on resized img
    everything = predictor(img_resized)

    overlay = img.copy().astype(np.float32)

    for ann in anns:
        x, y, w, h = ann['bbox']

        #scale bbox to resized image 
        x1 = int(x * scale_x) # This is important
        y1 = int(y * scale_y)
        x2 = int((x + w) * scale_x)
        y2 = int((y + h) * scale_y)

        box = [x1, y1, x2, y2]

        prompt_results = predictor.prompt(everything, bboxes=[box])

        if prompt_results is None:
            continue

        masks = prompt_results[0].masks.data.cpu().numpy()

        for m in masks:
            # mask is in resized space , bring back to original
            m = cv2.resize(m.astype(np.uint8), (W, H), interpolation=cv2.INTER_NEAREST).astype(bool)

            color = np.random.randint(0, 255, (3,), dtype=np.uint8)
            overlay[m] = overlay[m] * 0.6 + color * 0.4

        # draw original bbox (not scaled)
        cv2.rectangle(overlay, (int(x), int(y)), (int(x+w), int(y+h)), (0,255,0), 2)

    overlay = overlay.astype(np.uint8)

    save_path = f"fastsam_output/{img_info['file_name']}"
    cv2.imwrite(save_path, cv2.cvtColor(overlay, cv2.COLOR_RGB2BGR))

    print(f"Saved: {save_path}")
    count += 1

Saved: fastsam_output/000000397133.jpg
Saved: fastsam_output/000000037777.jpg
Saved: fastsam_output/000000252219.jpg
Saved: fastsam_output/000000087038.jpg
Saved: fastsam_output/000000174482.jpg
Saved: fastsam_output/000000403385.jpg
Saved: fastsam_output/000000006818.jpg
Saved: fastsam_output/000000480985.jpg
Saved: fastsam_output/000000458054.jpg
Saved: fastsam_output/000000331352.jpg
Saved: fastsam_output/000000296649.jpg
Saved: fastsam_output/000000386912.jpg
Saved: fastsam_output/000000502136.jpg
Saved: fastsam_output/000000491497.jpg
Saved: fastsam_output/000000184791.jpg
Saved: fastsam_output/000000348881.jpg
Saved: fastsam_output/000000289393.jpg
Saved: fastsam_output/000000522713.jpg
Saved: fastsam_output/000000181666.jpg
Saved: fastsam_output/000000017627.jpg
Saved: fastsam_output/000000143931.jpg
Saved: fastsam_output/000000303818.jpg
Saved: fastsam_output/000000463730.jpg
Saved: fastsam_output/000000460347.jpg
Saved: fastsam_output/000000322864.jpg
Saved: fastsam_output/000